In [1]:
import matplotlib.pyplot as plt
import os
import glob
import pandas_datareader.data as web
import datetime
import pandas as pd
from dbnomics import fetch_series
from utils import fetch_imf_data, fetch_imf_bulk
from config import country_dict, us_economy_metrics, country_dict_3_char


In [2]:
# Paths
imf_cpi_path = "data/imf_cpi.csv"
imf_gdp_path = "data/imf_gdp.csv"
imf_unemployment_path = "data/imf_unemployment_rate.csv"
imf_industrial_production_index_path = "data/imf_industrial_production_index.csv"
imf_exports_path = "data/imf_exports.csv"
imf_imports_path = "data/imf_imports.csv"
us_economy_path = "data/us_economy.csv"
cb_policy_rates_path = "data/cb_policy_rates.csv"
global_manufacturing_PMI_path = "data/global_manufacturing_PMI.csv"
im_ex_path = "data/im_ex/"
im_path = "data/im/"
ex_path = "data/ex/"
retail_sales_path = "data/retail_sales.csv"

target_countries = ["VN", "TH", "SG", "ID", "MY", "PH", "BN", "KH", "LA", "MM", "TL", "DE", "FR", "IT", "ES", "NL", "GB", "BE", "AT", "PT", "GR", "FI", "IE", "DK", "SE", "KW", "OM", "BH"]


In [5]:
# Pillar 1: System Health & Growth
# 1. REAL GDP GROWTH (Tăng trưởng GDP Thực tế - Hàng Quý 'Q' hoặc Hàng Năm 'A')
print("Fetching Real GDP Growth Data...")
imf_gdp = fetch_imf_data(
    country_dict=country_dict, 
    dataset_code = "IFS",
    metric_suffix="NGDP_R_XDC",
    frequency="A"
).dropna(subset=['value'])

# Tính YoY % trực tiếp trong Python nếu chuỗi dữ liệu gốc là dạng Số tuyệt đối
imf_gdp['value_yoy'] = imf_gdp.groupby('country_name')['value'].pct_change(periods=4) 
imf_gdp.to_csv(imf_gdp_path, index=False)

Fetching Real GDP Growth Data...


Could not load series: {'dataset_code': 'IFS', 'message': 'Could not load series', 'provider_code': 'IMF', 'series_code': 'A.VA.NGDP_R_XDC'}
Could not load series: {'dataset_code': 'IFS', 'message': 'Could not load series', 'provider_code': 'IMF', 'series_code': 'A.SX.NGDP_R_XDC'}
Could not load series: {'dataset_code': 'IFS', 'message': 'Could not load series', 'provider_code': 'IMF', 'series_code': 'A.VE.NGDP_R_XDC'}
Could not load series: {'dataset_code': 'IFS', 'message': 'Could not load series', 'provider_code': 'IMF', 'series_code': 'A.UZ.NGDP_R_XDC'}
Could not load series: {'dataset_code': 'IFS', 'message': 'Could not load series', 'provider_code': 'IMF', 'series_code': 'A.SUH.NGDP_R_XDC'}


In [6]:
# Pillar 1: System Health & Growth
# 2. INDUSTRIAL PRODUCTION INDEX - IPI (Thay đổi dataset_code="IFS")
ipi_raw = fetch_imf_bulk(
    dataset_code="IFS",
    metric_code="AIP_IX", 
    frequency="M", 
    country_list=target_countries
)
if not ipi_raw.empty:
    ipi_raw.dropna(subset=['value']).to_csv(imf_industrial_production_index_path, index=False)

--- Đang tải dữ liệu cho mã: AIP_IX (M) ---
✅ Tải thành công 11153 dòng dữ liệu!


In [7]:
# Pillar 2: Price Stability & Monetary Policy
# 3. CPI
imf_cpi = fetch_imf_data(country_dict=country_dict, dataset_code = "CPI", metric_suffix="PCPI_IX", frequency="M").dropna(subset=['value'])
# Show 10 samples
imf_cpi.head(10)
# Export data
imf_cpi.to_csv(imf_cpi_path)

Could not load series: {'dataset_code': 'CPI', 'message': 'Could not load series', 'provider_code': 'IMF', 'series_code': 'M.VA.PCPI_IX'}
Could not load series: {'dataset_code': 'CPI', 'message': 'Could not load series', 'provider_code': 'IMF', 'series_code': 'M.TM.PCPI_IX'}
Could not load series: {'dataset_code': 'CPI', 'message': 'Could not load series', 'provider_code': 'IMF', 'series_code': 'M.TV.PCPI_IX'}
Could not load series: {'dataset_code': 'CPI', 'message': 'Could not load series', 'provider_code': 'IMF', 'series_code': 'M.VU.PCPI_IX'}
Could not load series: {'dataset_code': 'CPI', 'message': 'Could not load series', 'provider_code': 'IMF', 'series_code': 'M.SUH.PCPI_IX'}
Could not load series: {'dataset_code': 'CPI', 'message': 'Could not load series', 'provider_code': 'IMF', 'series_code': 'M.U2.PCPI_IX'}


In [25]:
# Pillar 2: Price Stability & Monetary Policy
# 4. Central bank policy rates
cb_policy_rates = fetch_series(
    provider_code="BIS", 
    dataset_code="WS_CBPOL", 
    dimensions={
        "FREQ": ["D"],
        "REF_AREA": list(country_dict.keys())
    }
)
cb_policy_rates.to_csv(cb_policy_rates_path)

In [3]:
# Pillar 3: Labor Market & Consumer Market
# 5. Unemployment Rate
unemployment_raw = fetch_series(
    provider_code="IMF", 
    dataset_code="WEO:latest", 
    dimensions={
        "weo-country": list(country_dict_3_char.keys()),
        "weo-subject": ["LUR"],
        "unit": ["pcent_total_labor_force"]
    }
)
unemployment_raw.to_csv(imf_unemployment_path)

In [30]:
# Pillar 3: Labor Market & Consumer Market
# 6. Retail Sales Growth
# DBnomics: OECD (KEI).
retail_sales_raw_set_1 = fetch_series(
    provider_code="OECD", 
    dataset_code="MEI", 
    dimensions={
        "SUBJECT": ["SLRTTO02"],     #  Sales > Retail trade > Total retail trade > Value
        "MEASURE": ["IXEB"],    # US Dollars, monthly level
        "FREQUENCY": ["M"],     # Lấy theo Tháng (Monthly) để monitor cực nhạy
        "LOCATION": ['AUT', 'BEL', 'BGR', 'CHE', 'ESP', 'EST', 'FIN', 'FRA', 'GRC', 'HRV', 'HUN', 'IRL', 'ITA', 'LTU', 'NLD', 'NOR', 'PRT', 'SVK', 'TUR']
    }
)
retail_sales_raw_set_2 = fetch_series(
    provider_code="OECD", 
    dataset_code="MEI", 
    dimensions={
        "SUBJECT": ["SLRTTO02"],
        "MEASURE": ["ML"],
        "FREQUENCY": ["M"],
        "LOCATION": ['CHN', 'RUS', 'USA', 'ZAF']
    }
)
retail_sales_raw = pd.concat([retail_sales_raw_set_1, retail_sales_raw_set_2], ignore_index=True)
retail_sales_raw.to_csv(retail_sales_path, encoding="utf-8-sig")

In [5]:
# Pillar 4: Trade & Geopolitics
# 7. Global Manufacturing PMI
# DBnomics: WB (World Bank - Pink Sheet Commodity Prices) hoặc IMF.
global_manufacturing_PMI = fetch_series(
    provider_code="OECD", 
    dataset_code="MEI", 
    dimensions={
        "SUBJECT": ["BSCICP02"], 
        "FREQUENCY": ["M"],
    }
)
global_manufacturing_PMI.to_csv(global_manufacturing_PMI_path)

In [34]:
# Pillar 4: Trade & Geopolitics
# 8.1. Merchandise Trade Balance

# ==========================================
# 1. CẤU HÌNH & THIẾT LẬP
# ==========================================
# Đặt True khi muốn CẬP NHẬT DỮ LIỆU MỚI
# Đặt False khi bị CRASH/NGẮT MẠNG và muốn CHẠY NỐI TIẾP
FORCE_UPDATE = True  

im_ex_path = "data/im_ex"
os.makedirs(im_ex_path, exist_ok=True)

country_codes = list(country_dict.keys())
CHUNK_SIZE = 10

# ==========================================
# 2. TIẾN HÀNH TẢI DỮ LIỆU THEO CHUNK
# ==========================================
print(f"🚀 Bắt đầu quá trình tải (Chế độ FORCE_UPDATE = {FORCE_UPDATE})...\n")

for i in range(0, len(country_codes), CHUNK_SIZE):
    chunk_codes = country_codes[i : i + CHUNK_SIZE]
    batch_num = (i // CHUNK_SIZE) + 1
    file_path = os.path.join(im_ex_path, f"im_ex_batch_{batch_num}.csv")
    
    # Kiểm tra điều kiện bỏ qua nếu file đã tồn tại và không ép buộc update
    if not FORCE_UPDATE and os.path.exists(file_path):
        print(f"⏩ Nhóm {batch_num} ({len(chunk_codes)} nước) đã có file -> Bỏ qua.")
        continue

    print(f"🔄 Đang tải nhóm {batch_num}/{-(len(country_codes) // -CHUNK_SIZE)} ({len(chunk_codes)} nước): {chunk_codes}...")
    
    try:
        df_chunk = fetch_series(
            provider_code="IMF", 
            dataset_code="DOT",
            max_nb_series=53188,
            dimensions={
                "FREQ": ["M"],
                "REF_AREA": chunk_codes,
                "INDICATOR": ["TBG_USD"]
            }
        )
        
        if df_chunk is not None and not df_chunk.empty:
            df_chunk = df_chunk[df_chunk['value'].notna()]
            df_chunk.to_csv(file_path, index=False, encoding="utf-8-sig")
            print(f"   => ✅ Đã lưu: {file_path}")
        else:
            print(f"   => ⚠️ Nhóm {batch_num} không phản hồi dữ liệu.")
            
    except Exception as e:
        print(f"   => ❌ Lỗi ở nhóm {batch_num}: {e}")

print("\n🎉 Hoàn thành xong bước tải dữ liệu!")

🚀 Bắt đầu quá trình tải (Chế độ FORCE_UPDATE = True)...

🔄 Đang tải nhóm 1/8 (10 nước): ['VN', 'TH', 'SG', 'ID', 'MY', 'PH', 'BN', 'KH', 'LA', 'MM']...
   => ✅ Đã lưu: data/im_ex\im_ex_batch_1.csv
🔄 Đang tải nhóm 2/8 (10 nước): ['TL', 'DE', 'FR', 'IT', 'ES', 'NL', 'GB', 'BE', 'AT', 'PT']...
   => ✅ Đã lưu: data/im_ex\im_ex_batch_2.csv
🔄 Đang tải nhóm 3/8 (10 nước): ['GR', 'FI', 'IE', 'DK', 'SE', 'PL', 'CZ', 'RO', 'HU', 'VA']...
   => ✅ Đã lưu: data/im_ex\im_ex_batch_3.csv
🔄 Đang tải nhóm 4/8 (10 nước): ['UA', 'TR', 'XK', 'US', 'CA', 'MX', 'BR', 'AR', 'CO', 'SR']...
   => ✅ Đã lưu: data/im_ex\im_ex_batch_4.csv
🔄 Đang tải nhóm 5/8 (10 nước): ['SV', 'SX', 'TT', 'UY', 'VC', 'VE', 'SA', 'AE', 'QA', 'KW']...
   => ✅ Đã lưu: data/im_ex\im_ex_batch_5.csv
🔄 Đang tải nhóm 6/8 (10 nước): ['OM', 'BH', 'IQ', 'SY', 'TJ', 'TM', 'UZ', 'YE', 'SN', 'SO']...
   => ✅ Đã lưu: data/im_ex\im_ex_batch_6.csv
🔄 Đang tải nhóm 7/8 (10 nước): ['SS', 'SZ', 'TD', 'TG', 'TN', 'TZ', 'UG', 'ZA', 'ZM', 'ZW']...
   => ✅ 

In [35]:
# Pillar 4: Trade & Geopolitics
# 8.2. Merchandise Import Value

# ==========================================
# 1. CẤU HÌNH & THIẾT LẬP
# ==========================================
# Đặt True khi muốn CẬP NHẬT DỮ LIỆU MỚI
# Đặt False khi bị CRASH/NGẮT MẠNG và muốn CHẠY NỐI TIẾP
FORCE_UPDATE = True  

os.makedirs(im_path, exist_ok=True)

country_codes = list(country_dict.keys())
CHUNK_SIZE = 10

# ==========================================
# 2. TIẾN HÀNH TẢI DỮ LIỆU THEO CHUNK
# ==========================================
print(f"🚀 Bắt đầu quá trình tải (Chế độ FORCE_UPDATE = {FORCE_UPDATE})...\n")

for i in range(0, len(country_codes), CHUNK_SIZE):
    chunk_codes = country_codes[i : i + CHUNK_SIZE]
    batch_num = (i // CHUNK_SIZE) + 1
    file_path = os.path.join(im_path, f"im_batch_{batch_num}.csv")
    
    # Kiểm tra điều kiện bỏ qua nếu file đã tồn tại và không ép buộc update
    if not FORCE_UPDATE and os.path.exists(file_path):
        print(f"⏩ Nhóm {batch_num} ({len(chunk_codes)} nước) đã có file -> Bỏ qua.")
        continue

    print(f"🔄 Đang tải nhóm {batch_num}/{-(len(country_codes) // -CHUNK_SIZE)} ({len(chunk_codes)} nước): {chunk_codes}...")
    
    try:
        df_chunk = fetch_series(
            provider_code="IMF", 
            dataset_code="DOT",
            max_nb_series=53188,
            dimensions={
                "FREQ": ["M"],
                "REF_AREA": chunk_codes,
                "INDICATOR": ["TMG_FOB_USD"]
            }
        )
        
        if df_chunk is not None and not df_chunk.empty:
            df_chunk = df_chunk[df_chunk['value'].notna()]
            df_chunk.to_csv(file_path, index=False, encoding="utf-8-sig")
            print(f"   => ✅ Đã lưu: {file_path}")
        else:
            print(f"   => ⚠️ Nhóm {batch_num} không phản hồi dữ liệu.")
            
    except Exception as e:
        print(f"   => ❌ Lỗi ở nhóm {batch_num}: {e}")

print("\n🎉 Hoàn thành xong bước tải dữ liệu!")

🚀 Bắt đầu quá trình tải (Chế độ FORCE_UPDATE = True)...

🔄 Đang tải nhóm 1/8 (10 nước): ['VN', 'TH', 'SG', 'ID', 'MY', 'PH', 'BN', 'KH', 'LA', 'MM']...
   => ⚠️ Nhóm 1 không phản hồi dữ liệu.
🔄 Đang tải nhóm 2/8 (10 nước): ['TL', 'DE', 'FR', 'IT', 'ES', 'NL', 'GB', 'BE', 'AT', 'PT']...
   => ⚠️ Nhóm 2 không phản hồi dữ liệu.
🔄 Đang tải nhóm 3/8 (10 nước): ['GR', 'FI', 'IE', 'DK', 'SE', 'PL', 'CZ', 'RO', 'HU', 'VA']...
   => ✅ Đã lưu: data/im/im_batch_3.csv
🔄 Đang tải nhóm 4/8 (10 nước): ['UA', 'TR', 'XK', 'US', 'CA', 'MX', 'BR', 'AR', 'CO', 'SR']...
   => ✅ Đã lưu: data/im/im_batch_4.csv
🔄 Đang tải nhóm 5/8 (10 nước): ['SV', 'SX', 'TT', 'UY', 'VC', 'VE', 'SA', 'AE', 'QA', 'KW']...
   => ⚠️ Nhóm 5 không phản hồi dữ liệu.
🔄 Đang tải nhóm 6/8 (10 nước): ['OM', 'BH', 'IQ', 'SY', 'TJ', 'TM', 'UZ', 'YE', 'SN', 'SO']...
   => ⚠️ Nhóm 6 không phản hồi dữ liệu.
🔄 Đang tải nhóm 7/8 (10 nước): ['SS', 'SZ', 'TD', 'TG', 'TN', 'TZ', 'UG', 'ZA', 'ZM', 'ZW']...
   => ✅ Đã lưu: data/im/im_batch_7.csv
🔄

In [36]:
# Pillar 4: Trade & Geopolitics
# 8.3. Merchandise Export Value

# ==========================================
# 1. CẤU HÌNH & THIẾT LẬP
# ==========================================
# Đặt True khi muốn CẬP NHẬT DỮ LIỆU MỚI
# Đặt False khi bị CRASH/NGẮT MẠNG và muốn CHẠY NỐI TIẾP
FORCE_UPDATE = True  

os.makedirs(ex_path, exist_ok=True)

country_codes = list(country_dict.keys())
CHUNK_SIZE = 10

# ==========================================
# 2. TIẾN HÀNH TẢI DỮ LIỆU THEO CHUNK
# ==========================================
print(f"🚀 Bắt đầu quá trình tải (Chế độ FORCE_UPDATE = {FORCE_UPDATE})...\n")

for i in range(0, len(country_codes), CHUNK_SIZE):
    chunk_codes = country_codes[i : i + CHUNK_SIZE]
    batch_num = (i // CHUNK_SIZE) + 1
    file_path = os.path.join(ex_path, f"ex_batch_{batch_num}.csv")
    
    # Kiểm tra điều kiện bỏ qua nếu file đã tồn tại và không ép buộc update
    if not FORCE_UPDATE and os.path.exists(file_path):
        print(f"⏩ Nhóm {batch_num} ({len(chunk_codes)} nước) đã có file -> Bỏ qua.")
        continue

    print(f"🔄 Đang tải nhóm {batch_num}/{-(len(country_codes) // -CHUNK_SIZE)} ({len(chunk_codes)} nước): {chunk_codes}...")
    
    try:
        df_chunk = fetch_series(
            provider_code="IMF", 
            dataset_code="DOT",
            max_nb_series=53188,
            dimensions={
                "FREQ": ["M"],
                "REF_AREA": chunk_codes,
                "INDICATOR": ["TXG_FOB_USD"]
            }
        )
        
        if df_chunk is not None and not df_chunk.empty:
            df_chunk = df_chunk[df_chunk['value'].notna()]
            df_chunk.to_csv(file_path, index=False, encoding="utf-8-sig")
            print(f"   => ✅ Đã lưu: {file_path}")
        else:
            print(f"   => ⚠️ Nhóm {batch_num} không phản hồi dữ liệu.")
            
    except Exception as e:
        print(f"   => ❌ Lỗi ở nhóm {batch_num}: {e}")

print("\n🎉 Hoàn thành xong bước tải dữ liệu!")

🚀 Bắt đầu quá trình tải (Chế độ FORCE_UPDATE = True)...

🔄 Đang tải nhóm 1/8 (10 nước): ['VN', 'TH', 'SG', 'ID', 'MY', 'PH', 'BN', 'KH', 'LA', 'MM']...
   => ✅ Đã lưu: data/ex/ex_batch_1.csv
🔄 Đang tải nhóm 2/8 (10 nước): ['TL', 'DE', 'FR', 'IT', 'ES', 'NL', 'GB', 'BE', 'AT', 'PT']...
   => ✅ Đã lưu: data/ex/ex_batch_2.csv
🔄 Đang tải nhóm 3/8 (10 nước): ['GR', 'FI', 'IE', 'DK', 'SE', 'PL', 'CZ', 'RO', 'HU', 'VA']...
   => ✅ Đã lưu: data/ex/ex_batch_3.csv
🔄 Đang tải nhóm 4/8 (10 nước): ['UA', 'TR', 'XK', 'US', 'CA', 'MX', 'BR', 'AR', 'CO', 'SR']...
   => ✅ Đã lưu: data/ex/ex_batch_4.csv
🔄 Đang tải nhóm 5/8 (10 nước): ['SV', 'SX', 'TT', 'UY', 'VC', 'VE', 'SA', 'AE', 'QA', 'KW']...
   => ✅ Đã lưu: data/ex/ex_batch_5.csv
🔄 Đang tải nhóm 6/8 (10 nước): ['OM', 'BH', 'IQ', 'SY', 'TJ', 'TM', 'UZ', 'YE', 'SN', 'SO']...
   => ✅ Đã lưu: data/ex/ex_batch_6.csv
🔄 Đang tải nhóm 7/8 (10 nước): ['SS', 'SZ', 'TD', 'TG', 'TN', 'TZ', 'UG', 'ZA', 'ZM', 'ZW']...
   => ✅ Đã lưu: data/ex/ex_batch_7.csv
🔄 Đan

In [13]:
# Pillar 4: Trade & Geopolitics
# 9. Commodity Price Index
# DBnomics: WB (World Bank - Pink Sheet Commodity Prices) hoặc IMF.

In [14]:
# Timeframe covering Trump baseline through the entirety of the Biden administration
start = datetime.datetime(2000, 1, 1)
end = datetime.datetime(2025, 12, 31)

# Define the exact FRED series codes mapped to your portfolio naming convention
try:
    df_list = []
    for name, code in us_economy_metrics.items():
        print(f"-> Fetching {name} ({code})...")
        # Pull data directly from St. Louis Fed servers
        df = web.DataReader(code, 'fred', start, end)
        # 1. Clean the individual metric's index and rename columns
        df = df.reset_index().rename(columns={'DATE': 'Date', code: 'Value'})
        # 2. Handle frequency alignment: Forward-fill quarterly data before stacking
        # This ensures April and May inherit Q1 data before it gets mixed with monthly metrics
        if name in ['Real_GDP', 'Manufacturing_Investment']:
            # Create a continuous monthly date range to map the quarterly data onto
            monthly_range = pd.date_range(start=start, end=end, freq='MS')
            df = df.set_index('Date').reindex(monthly_range).ffill().reset_index().rename(columns={'index': 'Date'})
        # 3. Add the metadata column so we know which metric this row belongs to
        df['Metric_Name'] = name
        # Reorder columns to look clean: Date | Metric_Name | Value
        df = df[['Date', 'Metric_Name', 'Value']]        
        df_list.append(df)
    print("Consolidating and formatting data frequencies...")
    # Concatenate all tables along the Date axis
    bi_dataset = pd.concat(df_list, axis=0, ignore_index=True)
    # Forward-fill (ffill) the quarterly data (GDP & Investment) so monthly rows aren't blank
    # This prevents relationship errors inside Power BI's model
    bi_dataset = bi_dataset.ffill()
    # Reset index to turn the Date from an index into a normal clean column
    bi_dataset = bi_dataset.reset_index().rename(columns={'DATE': 'Date'})
    # Export to local project directory
    bi_dataset.to_csv(us_economy_path, index=False)
    print(f"Success! Master file saved as '{us_economy_path}'")
    print(bi_dataset.tail(10))

except Exception as e:
    print(f"Pipeline Execution Failed: {e}")

-> Fetching CPI_All_Items (CPIAUCSL)...
-> Fetching Unemployment_Rate (UNRATE)...
-> Fetching Total_Nonfarm_Payrolls (PAYEMS)...
-> Fetching Real_GDP (GDPC1)...
-> Fetching Manufacturing_Investment (C307RX1Q020SBEA)...
Consolidating and formatting data frequencies...
Success! Master file saved as 'data/us_economy.csv'
      index       Date               Metric_Name    Value
1550   1550 2025-03-01  Manufacturing_Investment  145.228
1551   1551 2025-04-01  Manufacturing_Investment  142.245
1552   1552 2025-05-01  Manufacturing_Investment  142.245
1553   1553 2025-06-01  Manufacturing_Investment  142.245
1554   1554 2025-07-01  Manufacturing_Investment  136.654
1555   1555 2025-08-01  Manufacturing_Investment  136.654
1556   1556 2025-09-01  Manufacturing_Investment  136.654
1557   1557 2025-10-01  Manufacturing_Investment  126.551
1558   1558 2025-11-01  Manufacturing_Investment  126.551
1559   1559 2025-12-01  Manufacturing_Investment  126.551
